# CFRM 521/421 Machine Learning for Finance
## Course Project Template

**Project Title:**

**Group Members:**  
- Qingyu (Wendy) Mao (email: qingym6@uw.edu, Algorithm: SVR)  
- Konoka Hamada (email: konokh@uw.edu, Algorithm: Baseline linear regression)  
- Xiran Zhang (email: xiran712@uw.edu, Algorithm: Decision tree)  
- Polina Loukiantchikov (email: ___, Algorithm: Regularized regression)  
- Name 5 (email: ___, Algorithm: ___)  

**Date:**


<span style="color:red">
**Please remove the bullet points in each section as you proceed and feel free to adjust the structure and contents as needed.
</span>

# 1. Introduction

## 1.1 Problem Statement
- What is the goal of your project?
- Why is this problem important in finance?
- What financial decision, prediction, or classification task are you studying?

## 1.2 Related Literature
- Briefly summarize relevant papers or prior work.
- Explain how your project relates to existing studies.
- Cite all sources properly.

## 1.3 Contribution
- What does your project add beyond existing work?
- Is your contribution empirical comparison, replication with extensions, new data, or a new application?


# 2. Data Description

## 2.1 Data Source
- Describe the original source of the data.
- Include links, API names, or repository names if relevant.

## 2.2 Data Structure
- What does each row represent?
- What is the sampling frequency?
- How many observations are there?
- What time period does the data cover?

## 2.3 Target Variable
- Define the response variable clearly.

## 2.4 Features
- List and briefly describe the predictors.
- Give a few examples.

## 2.5 Data Cleaning and Preprocessing
- Missing values
- Outlier handling
- Scaling or normalization
- Feature engineering
- Train/validation/test split


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reproducibility
np.random.seed(42)


In [ ]:
# Load your data here
# Example:
# df = pd.read_csv('your_data.csv')
# df.head()

url = "https://raw.githubusercontent.com/p0linal1/CFRM-421-Project/refs/heads/main/kc_house_data.csv"
df = pd.read_csv(url)

df.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [ ]:
# For 2.2

# Check the structure of the dataset
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns)

# Convert date column to datetime and check the time period
df["date"] = pd.to_datetime(df["date"])

print("\nEarliest sale date:", df["date"].min())
print("Latest sale date:", df["date"].max())

# Check target variable
print("\nTarget variable summary:")
print(df["price"].describe())

# Check missing values
print("\nMissing values by column:")
print(df.isnull().sum())

Dataset shape: (21613, 21)

Column names:
Index(['id', 'date', 'price', 'bedrooms', 'bathrooms', 'sqft_living',
       'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade',
       'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode',
       'lat', 'long', 'sqft_living15', 'sqft_lot15'],
      dtype='object')

Earliest sale date: 2014-05-02 00:00:00
Latest sale date: 2015-05-27 00:00:00

Target variable summary:
count    2.161300e+04
mean     5.400881e+05
std      3.671272e+05
min      7.500000e+04
25%      3.219500e+05
50%      4.500000e+05
75%      6.450000e+05
max      7.700000e+06
Name: price, dtype: float64

Missing values by column:
id               0
date             0
price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
grade            0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
zipcode          0
lat

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import pandas as pd
import numpy as np

# copy of the original dataframe
data = df.copy()

# convert sale date and create date-based feature
data["date"] = pd.to_datetime(data["date"])
data["sale_year"] = data["date"].dt.year
data["sale_month"] = data["date"].dt.month

data["was_renovated"] = (data["yr_renovated"] > 0).astype(int)

# Target variable
y = data["price"]

# Features for all models
features = [
    "bedrooms",
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "floors",
    "waterfront",
    "view",
    "condition",
    "grade",
    "sqft_above",
    "sqft_basement",
    "yr_built",
    "was_renovated",
    "zipcode",
    "lat",
    "long",
    "sale_year",
    "sale_month"
]

X = data[features]

# zipcode is treated as categorical
categorical_features = ["zipcode"]
numeric_features = [col for col in features if col not in categorical_features]

# preprocessing for the models
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("Number of numerical features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

Training set shape: (17290, 18)
Test set shape: (4323, 18)
Number of numerical features: 17
Number of categorical features: 1


# 3. Exploratory Data Analysis

- Summarize the data.
- Visualize key variables.
- Examine the target variable.
- Discuss stylized facts or empirical patterns relevant to the financial problem.


In [ ]:
# Basic summary statistics
# df.describe()


In [ ]:
# Example visualization
# plt.figure(figsize=(6,4))
# df['your_column'].hist(bins=30)
# plt.title('Distribution of your_column')
# plt.show()


# 4. Methodology

Clearly separate the problem description from the learning algorithms.

Include a separate subsection for hyperparameter tuning:
- Explain how tuning is performed.
- Make the comparison fair across models.
- State the validation procedure clearly.

## 4.1 Overview of Models
- You must try at least as many algorithms as group members.
- Each group member should implement at least one algorithm.
- Use models within the scope of the course.
- If using a more advanced model, provide sufficient background and compare it against standard baselines first.


## 4.2 Model 1: Basic Linear Regression

**Implemented by:**

- Motivation
- Model description
- Why this method is appropriate/Key assumptions


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)


linear_model.fit(X_train, y_train)


y_pred_linear = linear_model.predict(X_test)


rmse_linear = np.sqrt(mean_squared_error(y_test, y_pred_linear))
mae_linear = mean_absolute_error(y_test, y_pred_linear)

print("Baseline Linear Regression Results")
print(f"RMSE: ${rmse_linear:,.2f}")
print(f"MAPE: {mape_linear:.2f}%")

Baseline Linear Regression Results
RMSE: $170,382.21
MAPE: 19.47%


## 4.3 Model 2: ___

**Implemented by:**

- Motivation
- Model description
- Why this method is appropriate

## 4.4 Model 3: ___

**Implemented by:**

- Motivation
- Model description
- Why this method is appropriate

## 4.5 Model 4: ___

**Implemented by:**

- Motivation
- Model description
- Why this method is appropriate

# 5. Results

Clearly separate the presentation of results from the conclusions.

## 5.1 Evaluation Metrics
- Explain why the chosen metrics are appropriate.

## 5.2 Main Quantitative Results
- Present results in tables. Compare model performance after tuning.


## 5.3 Visualizations
- Prediction vs actual
- Residual plots
- Feature importance
- Confusion matrix or ROC curve if classification


# 6. Discussions and Conclusions

Make it brief; (2-3 paragraphs max)

Discuss:
- Which model performed best?
- Why do you think it performed best?
- What do the results mean in the financial context?
- Are there economic or practical implications?
- What are the limitations of the study?

Conclude:
- Summarize the main findings.
- State the major takeaway.
- Suggest possible future work.

# Appendix. Reproducibility

- State the software environment.
- State package versions if relevant.
- Explain how to reproduce the analysis.
- Ensure the notebook has been run from start to finish.


In [ ]:
# example: package versions
# import sys
# print(sys.version)
# print(pd.__version__)
# print(np.__version__)

# References

- Include all papers, datasets, websites, and software packages cited in the notebook.
- Use a consistent citation style.
